In [17]:
%pip install scikit-learn pandas numpy optuna xgboost lightgbm catboost imbalanced-learn


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
# %%
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    # original features
    df['prev_success']        = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted']     = (df['previous'] == 0).astype(int)
    df['pdays_clean']         = df['pdays'].apply(lambda x: 999 if x == -1 else x)
    df['prev_contacts_log']   = np.log1p(df['previous'])
    df['duration_log']        = np.log1p(df['duration'])
    df['log_balance']         = np.log1p(df['balance'].clip(lower=0))
    df['is_debt']             = (df['balance'] < 0).astype(int)
    df['log_campaign']        = np.log1p(df['campaign'])
    df['month_sin']           = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']           = np.cos(2 * np.pi * df['month'] / 12)
    df['long_call']           = (df['duration'] > 300).astype(int)
    df['long_call_x_success'] = df['long_call'] * df['prev_success']
    df['duration_x_prev']     = df['duration_log'] * df['prev_contacts_log']
    # v4 interaction features
    df['duration_x_success']  = df['duration_log'] * df['prev_success']
    df['pdays_x_prev']        = (1 / (df['pdays_clean'] + 1)) * df['prev_contacts_log']
    df['balance_x_debt']      = df['log_balance'] * (1 - df['is_debt'])
    df['duration_per_contact']= df['duration_log'] / (df['log_campaign'] + 1)
    df['age_young']           = (df['age'] < 30).astype(int)
    df['age_senior']          = (df['age'] >= 55).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)
print('TRAIN_DATA shape:', TRAIN_DATA.shape)
print('TEST_DATA  shape:', TEST_DATA.shape)

TRAIN_DATA shape: (29839, 35)
TEST_DATA  shape: (19893, 35)


In [19]:
# %%
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = [c for c in TRAIN_DATA.columns if c not in cat_cols]

ENCODER = OrdinalEncoder(
    handle_unknown='use_encoded_value', unknown_value=-1
).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]), columns=cat_cols, index=df.index
    )
    return pd.concat([cat_enc, df[num_cols].copy()], axis=1)

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values

def target_encode_cv(X_tr, y_tr, X_te, cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc = X_tr.copy()
    X_te_enc = X_te.copy()
    global_mean = y_tr.mean()
    for col in cols:
        oof     = np.full(len(X_tr), global_mean)
        te_vals = np.zeros(len(X_te))
        for fold_tr_idx, fold_val_idx in skf.split(X_tr, y_tr):
            means = y_tr.iloc[fold_tr_idx].groupby(X_tr[col].iloc[fold_tr_idx]).mean()
            oof[fold_val_idx] = X_tr[col].iloc[fold_val_idx].map(means).fillna(global_mean).values
            te_vals += X_te[col].reset_index(drop=True).map(means).fillna(global_mean).values / n_splits
        X_tr_enc[col + '_te'] = oof
        X_te_enc[col + '_te'] = te_vals
    return X_tr_enc, X_te_enc

y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)
print('X_train_te shape:', X_train_te.shape)
print('X_test_te  shape:', X_test_te.shape)

X_train_te shape: (29839, 43)
X_test_te  shape: (19893, 43)


In [20]:
# %%
# ================================================================
#  BEST PARAMS FROM v4 OPTUNA RUN  — no retuning needed!
#  just paste results directly, saves 4 hours :)
# ================================================================
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

best_xgb = {
    'n_estimators'     : 1398,
    'learning_rate'    : 0.014688107933307866,
    'max_depth'        : 9,
    'min_child_weight' : 11,
    'subsample'        : 0.9081675632335575,
    'colsample_bytree' : 0.9154397069234146,
    'colsample_bylevel': 0.7859644524805499,
    'reg_alpha'        : 14.449527198623775,
    'reg_lambda'       : 0.0013131988451547652,
    'gamma'            : 1.3271607503475447,
    'scale_pos_weight' : scale_pos,
    'eval_metric'      : 'logloss',
    'n_jobs'           : -1,
}

best_lgbm = {
    'boosting_type'    : 'gbdt',
    'n_estimators'     : 1544,
    'learning_rate'    : 0.009776236382241175,
    'max_depth'        : 12,
    'num_leaves'       : 189,
    'min_child_samples': 79,
    'subsample'        : 0.6758166608075022,
    'colsample_bytree' : 0.6287663520283125,
    'reg_alpha'        : 14.059097249147474,
    'reg_lambda'       : 0.5598467566307593,
    'class_weight'     : 'balanced',
    'n_jobs'           : -1,
    'verbose'          : -1,
}

best_cat = {
    'iterations'          : 639,
    'learning_rate'       : 0.013197977619438363,
    'depth'               : 10,
    'l2_leaf_reg'         : 9.52653108639879,
    'bagging_temperature' : 1.422724713850696,
    'random_strength'     : 0.14507304848953856,
    'border_count'        : 167,
    'auto_class_weights'  : 'Balanced',
    'eval_metric'         : 'Logloss',
    'verbose'             : 0,
}

print('Params loaded from v4 optuna run ✓')
print(f'scale_pos_weight: {scale_pos:.4f}')

Params loaded from v4 optuna run ✓
scale_pos_weight: 7.5597


In [21]:
# %%
# ================================================================
#  OOF + TEST PREDICTIONS
#  seed averaging: each model runs with 3 seeds → averaged
#  adds diversity without any extra tuning, ~30-40 mins total
# ================================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_curve

N_SPLITS = 10
SEEDS    = [42, 2, 125]   # 3 seeds per model for averaging

model_names = ['XGB', 'LGBM', 'CAT']
X_arr    = X_train_te.values
X_te_arr = X_test_te.values

# accumulate across seeds then divide
oof_preds  = {name: np.zeros(len(y_train))   for name in model_names}
test_preds = {name: np.zeros(len(X_test_te)) for name in model_names}

for seed in SEEDS:
    print(f'\n--- Seed {seed} ---')
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    oof_seed  = {name: np.zeros(len(y_train))   for name in model_names}
    test_seed = {name: np.zeros(len(X_test_te)) for name in model_names}

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
        X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        # XGB
        m = XGBClassifier(**best_xgb, random_state=seed)
        m.fit(X_tr, y_tr)
        oof_seed['XGB'][val_idx]  = m.predict_proba(X_val)[:, 1]
        test_seed['XGB']         += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        # LGBM
        m = LGBMClassifier(**best_lgbm, random_state=seed)
        m.fit(X_tr, y_tr)
        oof_seed['LGBM'][val_idx] = m.predict_proba(X_val)[:, 1]
        test_seed['LGBM']        += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        # CatBoost
        m = CatBoostClassifier(**best_cat, random_seed=seed)
        m.fit(X_tr, y_tr)
        oof_seed['CAT'][val_idx]  = m.predict_proba(X_val)[:, 1]
        test_seed['CAT']         += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        print(f'  Fold {fold+1}/{N_SPLITS} done')

    # accumulate
    for name in model_names:
        oof_preds[name]  += oof_seed[name]  / len(SEEDS)
        test_preds[name] += test_seed[name] / len(SEEDS)

print('\nOOF Balanced Accuracy per model (Youden J threshold):')
for name in model_names:
    fpr, tpr, thresholds = roc_curve(y_train, oof_preds[name])
    best_t  = float(thresholds[np.argmax(tpr - fpr)])
    best_ba = balanced_accuracy_score(y_train, (oof_preds[name] >= best_t).astype(int))
    print(f'  {name:5s}: BA={best_ba:.5f}  threshold={best_t:.4f}')


--- Seed 42 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

--- Seed 2 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

--- Seed 125 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

OOF Balanced Accuracy per model (Youden J threshold):
  XGB  : BA=0.87533  threshold=0.3319
  LGBM : BA=0.87556  threshold=0.3973
  CAT  : BA=0.87485  threshold=0.3912


In [22]:
# %%
# ================================================================
#  ENSEMBLE  —  simple equal-weight average
#  all 3 models are within 0.001 OOF BA of each other
#  equal weights maximises diversity benefit
# ================================================================
from sklearn.metrics import balanced_accuracy_score, roc_curve
from scipy.stats import rankdata

def rank_norm(arr):
    return rankdata(arr) / len(arr)

# equal weights — no amplification, just diversity
weights = np.array([1/3, 1/3, 1/3])
print('Model weights (equal):')
for n, w in zip(model_names, weights):
    print(f'  {n}: {w:.4f}')

oof_rank  = np.column_stack([rank_norm(oof_preds[n])  for n in model_names])
test_rank = np.column_stack([rank_norm(test_preds[n]) for n in model_names])

oof_blend  = oof_rank  @ weights
test_blend = test_rank @ weights

# Youden J threshold
fpr_b, tpr_b, thresholds_b = roc_curve(y_train, oof_blend)
j_b            = tpr_b - fpr_b
best_threshold = float(thresholds_b[np.argmax(j_b)])
best_ba        = balanced_accuracy_score(
    y_train, (oof_blend >= best_threshold).astype(int)
)

print(f'\nBlended OOF BA    : {best_ba:.5f}')
print(f'Optimal threshold : {best_threshold:.4f}')
print(f'Target to beat    : 0.87600 OOF  →  ~0.88443 LB')

# threshold neighbourhood
print('\nThreshold | #Pred-1 | OOF BA')
print('-' * 38)
for t in np.arange(
    max(0.01, best_threshold - 0.05),
    min(0.99, best_threshold + 0.06),
    0.005
):
    preds = (oof_blend >= t).astype(int)
    ba    = balanced_accuracy_score(y_train, preds)
    mark  = ' <- best' if abs(t - best_threshold) < 0.003 else ''
    print(f'  {t:.3f}  |  {preds.sum():6d}  |  {ba:.4f}{mark}')

Model weights (equal):
  XGB: 0.3333
  LGBM: 0.3333
  CAT: 0.3333

Blended OOF BA    : 0.87627
Optimal threshold : 0.7618
Target to beat    : 0.87600 OOF  →  ~0.88443 LB

Threshold | #Pred-1 | OOF BA
--------------------------------------
  0.712  |    8567  |  0.8697
  0.717  |    8413  |  0.8705
  0.722  |    8277  |  0.8724
  0.727  |    8125  |  0.8730
  0.732  |    7956  |  0.8743
  0.737  |    7804  |  0.8750
  0.742  |    7646  |  0.8754
  0.747  |    7503  |  0.8749
  0.752  |    7359  |  0.8754
  0.757  |    7231  |  0.8750
  0.762  |    7097  |  0.8763 <- best
  0.767  |    6939  |  0.8747
  0.772  |    6775  |  0.8734
  0.777  |    6637  |  0.8730
  0.782  |    6499  |  0.8727
  0.787  |    6353  |  0.8715
  0.792  |    6197  |  0.8700
  0.797  |    6036  |  0.8686
  0.802  |    5900  |  0.8665
  0.807  |    5751  |  0.8635
  0.812  |    5591  |  0.8599
  0.817  |    5451  |  0.8568


In [23]:
# %%
# ================================================================
#  DECISION GATE  —  only submit if OOF > 0.876
# ================================================================
SUBMIT_THRESHOLD = 0.876

if best_ba >= SUBMIT_THRESHOLD:
    print(f'✅ OOF {best_ba:.5f} >= {SUBMIT_THRESHOLD} — WORTH SUBMITTING')
else:
    print(f'❌ OOF {best_ba:.5f} < {SUBMIT_THRESHOLD} — DO NOT SUBMIT, investigate further')

print(f'\nExpected LB ~ {best_ba + 0.00843:.5f}')
print(f'Need to beat   0.88566 (2nd place)')

✅ OOF 0.87627 >= 0.876 — WORTH SUBMITTING

Expected LB ~ 0.88470
Need to beat   0.88566 (2nd place)


In [24]:
# %%
# ================================================================
#  GENERATE SUBMISSION
# ================================================================
test_classes = (test_blend >= best_threshold).astype(int)
n1 = test_classes.sum()
n0 = (test_classes == 0).sum()

print(f'Prediction distribution — 0: {n0}, 1: {n1} (pos-rate {n1/(n0+n1)*100:.1f}%)')
print(f'Blended OOF BA  : {best_ba:.5f}')
print(f'Threshold used  : {best_threshold:.4f}')

submission = pd.DataFrame({
    'id'          : TEST_DATA.index,
    'subscription': test_classes
})
submission.to_csv('submission_optuna_v5.csv', index=False)
print('\nSaved submission_optuna_v5.csv ✓')
print(submission.head())
print(submission['subscription'].value_counts())

Prediction distribution — 0: 15168, 1: 4725 (pos-rate 23.8%)
Blended OOF BA  : 0.87627
Threshold used  : 0.7618

Saved submission_optuna_v5.csv ✓
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             1
subscription
0    15168
1     4725
Name: count, dtype: int64
